### Tech Challenge — Previsão de Obesidade
#### Notebook 02 — Pré-processamento e Pipeline

**Status:** EDA concluída no notebook 01. Decisões fundamentadas.
**Objetivo desta etapa:** transformar as decisões da EDA em uma pipeline reproduzível, modular e pronta para treinamento.

**Princípio:** este notebook NÃO contém lógica de transformação. Tudo está em `/src`.

---

---
### 1. Setup, imports e configuração de path.

In [1]:
# Adiciona a raiz do projeto ao sys.path para importar de src/
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT: {PROJECT_ROOT}')

PROJECT_ROOT: c:\Users\leona\OneDrive\Desktop\Prova FIAP\2techchallange\obesity_project


In [2]:
# Imports do projeto (toda a lógica vem de src/)
from src.utils.config import (
    RANDOM_STATE, TARGET_COL, CLASS_ORDER, ARTIFACTS_DIR, PROCESSED_DATA_DIR,
    TARGET_ENCODER_FILENAME, ensure_dirs,
)
from src.data import load_and_validate, stratified_split
from src.features import (
    drop_exact_duplicates,
    build_pipeline,
    get_feature_names_after_transform,
    OrdinalTargetEncoder,
)

# Bibliotecas externas
import numpy as np
import pandas as pd
import joblib

# Sanity: garantir que os diretórios existem
ensure_dirs()

print(f'RANDOM_STATE = {RANDOM_STATE}')
print(f'CLASS_ORDER = {CLASS_ORDER}')

RANDOM_STATE = 42
CLASS_ORDER = ['Insufficient_Weight', 'Normal_Weight', 'Overweight_Level_I', 'Overweight_Level_II', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III']


---
#### 2. Carregamento e validação de schema.

Toda a lógica está em `src/data/loader.py`. Este notebook apenas chama `load_and_validate()`, que combina carga + validação de schema.

Se o schema estiver fora do esperado, levanta `SchemaValidationError` imediatamente, falha cedo, evita propagar inconsistência.

In [3]:
df = load_and_validate()
print(f'Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'Schema OK ✓')
df.head(3)

Shape: 2,111 linhas × 17 colunas
Schema OK ✓


,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight


---
#### 3. Limpeza training-only, remoção de duplicatas exatas.

`drop_exact_duplicates` é **training-only** (altera número de linhas, não pode ir dentro de um sklearn Pipeline). Aplicado ANTES do split para evitar que registros sintéticos idênticos vazem entre treino e teste.

In [4]:
n_before = len(df)
df_clean = drop_exact_duplicates(df)
n_after = len(df_clean)

print(f'Antes: {n_before:,} linhas')
print(f'Depois: {n_after:,} linhas')
print(f'Removidas: {n_before - n_after} duplicatas exatas')

Antes: 2,111 linhas
Depois: 2,087 linhas
Removidas: 24 duplicatas exatas


---
#### 4. Split estratificado 70/15/15.

`stratified_split` faz o split em dois passos: primeiro separa 15% para teste (isolado a partir deste ponto), depois divide o restante em treino (70% do total) e validação (15% do total).

- `RANDOM_STATE=42` fixo para reprodutibilidade
- Estratificação no target garante distribuição de classes proporcional em cada split

In [5]:
splits = stratified_split(df_clean)
print(splits.summary().to_string(index=False))

split  n_rows  pct_total
train    1460  69.956876
  val     313  14.997604
 test     314  15.045520


In [6]:
# Validação: distribuição do target nos três splits deve ser próxima (estratificação)
dist_table = pd.DataFrame({
    'train': splits.y_train.value_counts(normalize=True),
    'val':   splits.y_val.value_counts(normalize=True),
    'test':  splits.y_test.value_counts(normalize=True),
}).reindex(CLASS_ORDER).round(4) * 100

print('Distribuição (%) do target por split:')
print(dist_table.to_string())

# Sanity: diferença máxima entre splits para qualquer classe
max_diff = (dist_table.max(axis=1) - dist_table.min(axis=1)).max()
print(f'\nMaior diferença entre splits para uma única classe: {max_diff:.2f} pp')
print('(Se < 1.5 pp, a estratificação está funcionando corretamente.)')

Distribuição (%) do target por split:
                     train    val   test
Obesity                                 
Insufficient_Weight  12.81  12.78  12.74
Normal_Weight        13.56  13.42  13.38
Overweight_Level_I   13.29  13.10  13.06
Overweight_Level_II  13.90  13.74  14.01
Obesity_Type_I       16.78  16.93  16.88
Obesity_Type_II      14.18  14.38  14.33
Obesity_Type_III     15.48  15.65  15.61

Maior diferença entre splits para uma única classe: 0.27 pp
(Se < 1.5 pp, a estratificação está funcionando corretamente.)


---
#### 5. Encoding do target — `OrdinalTargetEncoder`

Não usei `sklearn.LabelEncoder` porque ele ordena classes alfabeticamente ao fitar, o que invertaria a ordem clínica (`Obesity_Type_I` viria antes de `Overweight_Level_I` por O < O).

`OrdinalTargetEncoder` mantém a ordem clínica definida em `config.CLASS_ORDER`. Interface compatível com LabelEncoder (`fit`, `transform`, `inverse_transform`).

In [7]:
target_encoder = OrdinalTargetEncoder()
target_encoder.fit(splits.y_train)

# Mapeamento clínico
print('Mapeamento classe → inteiro (ordem clínica):')
for i, cls in enumerate(target_encoder.classes_):
    print(f'  {i} → {cls}')

# Transforma os três splits
y_train_int = target_encoder.transform(splits.y_train)
y_val_int   = target_encoder.transform(splits.y_val)
y_test_int  = target_encoder.transform(splits.y_test)

# Sanity: roundtrip
sample = splits.y_train.iloc[:5].tolist()
roundtrip = target_encoder.inverse_transform(target_encoder.transform(sample))
assert list(roundtrip) == sample, 'Roundtrip falhou!'
print(f'\n✓ Roundtrip OK: {sample} → ints → {list(roundtrip)}')

Mapeamento classe → inteiro (ordem clínica):
  0 → Insufficient_Weight
  1 → Normal_Weight
  2 → Overweight_Level_I
  3 → Overweight_Level_II
  4 → Obesity_Type_I
  5 → Obesity_Type_II
  6 → Obesity_Type_III

✓ Roundtrip OK: ['Overweight_Level_I', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_I', 'Insufficient_Weight'] → ints → [np.str_('Overweight_Level_I'), np.str_('Obesity_Type_III'), np.str_('Overweight_Level_I'), np.str_('Overweight_Level_I'), np.str_('Insufficient_Weight')]



#### 6. Construção das pipelines sklearn Modelo A e Modelo B.

In [8]:
# Modelo A — diagnóstico (com Weight + IMC)
pipe_a = build_pipeline(model_type='A', scale_continuous=False)

# Modelo B — triagem comportamental (sem Weight, sem IMC)
pipe_b = build_pipeline(model_type='B', scale_continuous=False)

print('Pipeline Modelo A:')
print('  Steps:', [name for name, _ in pipe_a.steps])
print()
print('Pipeline Modelo B:')
print('  Steps:', [name for name, _ in pipe_b.steps])

Pipeline Modelo A:
  Steps: ['prepare', 'add_bmi', 'preprocessor']

Pipeline Modelo B:
  Steps: ['prepare', 'preprocessor']


---
#### 7. Fit em treino e inspeção do output.

- O Modelo A tem exatamente 2 colunas a mais que o Modelo B (Weight, IMC)

In [9]:
# Fit nas pipelines (apenas treino — val e test ficam intocados)
pipe_a.fit(splits.X_train)
pipe_b.fit(splits.X_train)

# Transform o treino para inspecionar dimensões
X_train_a = pipe_a.transform(splits.X_train)
X_train_b = pipe_b.transform(splits.X_train)

print(f'Modelo A — output shape: {X_train_a.shape}')
print(f'Modelo B — output shape: {X_train_b.shape}')
print(f'Diferença de colunas: {X_train_a.shape[1] - X_train_b.shape[1]} (deve ser 2: Weight + IMC)')

# Sanity: zero NaN
assert not np.isnan(X_train_a).any(), 'NaN encontrado em Modelo A!'
assert not np.isnan(X_train_b).any(), 'NaN encontrado em Modelo B!'
print('\n✓ Zero NaN em ambas as pipelines')

Modelo A — output shape: (1460, 20)
Modelo B — output shape: (1460, 18)
Diferença de colunas: 2 (deve ser 2: Weight + IMC)

✓ Zero NaN em ambas as pipelines


In [10]:
# Feature names após encoding completo
features_a = get_feature_names_after_transform(pipe_a)
features_b = get_feature_names_after_transform(pipe_b)

print(f'Modelo A — {len(features_a)} features:')
for f in features_a:
    print(f'  • {f}')
print()
print(f'Modelo B — {len(features_b)} features:')
for f in features_b:
    print(f'  • {f}')

# As features exclusivas do Modelo A devem ser Weight e IMC
exclusive_a = set(features_a) - set(features_b)
print(f'\nFeatures exclusivas do Modelo A: {sorted(exclusive_a)}')
assert exclusive_a == {'Weight', 'IMC'}, f'Inesperado: {exclusive_a}'
print('✓ Modelo A excede Modelo B exatamente em Weight + IMC')

Modelo A — 20 features:
  • Age
  • Height
  • Weight
  • IMC
  • Gender
  • family_history
  • FAVC
  • SMOKE
  • SCC
  • CAEC
  • CALC
  • FCVC
  • NCP
  • CH2O
  • FAF
  • TUE
  • MTRANS_Bike
  • MTRANS_Motorbike
  • MTRANS_Public_Transportation
  • MTRANS_Walking

Modelo B — 18 features:
  • Age
  • Height
  • Gender
  • family_history
  • FAVC
  • SMOKE
  • SCC
  • CAEC
  • CALC
  • FCVC
  • NCP
  • CH2O
  • FAF
  • TUE
  • MTRANS_Bike
  • MTRANS_Motorbike
  • MTRANS_Public_Transportation
  • MTRANS_Walking

Features exclusivas do Modelo A: ['IMC', 'Weight']
✓ Modelo A excede Modelo B exatamente em Weight + IMC


---
### 8. Aplicação em val/test, verificação de coerência.


In [11]:
X_val_a   = pipe_a.transform(splits.X_val)
X_test_a  = pipe_a.transform(splits.X_test)
X_val_b   = pipe_b.transform(splits.X_val)
X_test_b  = pipe_b.transform(splits.X_test)

print('Shape consistency check:')
print(f'  Modelo A:  train {X_train_a.shape}  val {X_val_a.shape}  test {X_test_a.shape}')
print(f'  Modelo B:  train {X_train_b.shape}  val {X_val_b.shape}  test {X_test_b.shape}')

# Verificação adicional: categorias MTRANS no val/test estão todas representadas em train?
mtrans_train = set(splits.X_train['MTRANS'].unique())
mtrans_val   = set(splits.X_val['MTRANS'].unique())
mtrans_test  = set(splits.X_test['MTRANS'].unique())
new_in_val   = mtrans_val - mtrans_train
new_in_test  = mtrans_test - mtrans_train
print(f'\nMTRANS novas em val: {new_in_val if new_in_val else "nenhuma ✓"}')
print(f'MTRANS novas em test: {new_in_test if new_in_test else "nenhuma ✓"}')

Shape consistency check:
  Modelo A:  train (1460, 20)  val (313, 20)  test (314, 20)
  Modelo B:  train (1460, 18)  val (313, 18)  test (314, 18)

MTRANS novas em val: nenhuma ✓
MTRANS novas em test: nenhuma ✓


---
#### 9. Verificação de determinismo, coerência entre execuções.

Aplicar `transform` na mesma entrada duas vezes deve dar exatamente o mesmo resultado. Esta é a garantia mais importante da pipeline: inferência em produção será idêntica à do treino, sem drift.

In [12]:
# Transforma o mesmo conjunto duas vezes
out1 = pipe_a.transform(splits.X_val)
out2 = pipe_a.transform(splits.X_val)
assert np.array_equal(out1, out2), 'NÃO-DETERMINÍSTICO!'
print('✓ Determinismo: pipe_a.transform produz output idêntico em chamadas repetidas')

✓ Determinismo: pipe_a.transform produz output idêntico em chamadas repetidas


---
#### 10. Simulação de inferência, raw input do formulário.

Streamlit enviará: uma única linha com strings (`yes`/`no`, `Female`/`Male`, etc.). A pipeline aceita esse input sem qualquer pré-processamento manual — toda a transformação (arredondamento, mapeamento, IMC, one-hot) acontece internamente.

In [13]:
# Paciente fictício preenchendo o formulário do app
single_patient = pd.DataFrame([{
    'Gender':         'Female',
    'Age':            27,                       # inteiro (form discreto)
    'Height':         1.65,
    'Weight':         65.0,
    'family_history': 'yes',
    'FAVC':           'yes',
    'FCVC':           2,                        # escala 1-3
    'NCP':            3,
    'CAEC':           'Sometimes',
    'SMOKE':          'no',
    'CH2O':           2,
    'SCC':            'no',
    'FAF':            1,
    'TUE':            1,
    'CALC':           'Sometimes',
    'MTRANS':         'Public_Transportation',
}])

print('Raw input (1 linha) que o app enviaria:')
print(single_patient.T)

# Pipeline transforma direto, sem chamadas manuais
out_a = pipe_a.transform(single_patient)
out_b = pipe_b.transform(single_patient)

print(f'\nOutput Modelo A: shape {out_a.shape}')
print(f'Output Modelo B: shape {out_b.shape}')

# IMC calculado internamente
imc_expected = single_patient['Weight'].iloc[0] / single_patient['Height'].iloc[0] ** 2
imc_index = features_a.index('IMC')
print(f'\nIMC esperado: {imc_expected:.4f}')
print(f'IMC calculado pela pipeline: {out_a[0, imc_index]:.4f}')
assert abs(out_a[0, imc_index] - imc_expected) < 1e-9, 'IMC interno divergente!'
print('✓ IMC calculado corretamente dentro da pipeline')

Raw input (1 linha) que o app enviaria:
                                    0
Gender                         Female
Age                                27
Height                           1.65
Weight                           65.0
family_history                    yes
FAVC                              yes
FCVC                                2
NCP                                 3
CAEC                        Sometimes
SMOKE                              no
CH2O                                2
SCC                                no
FAF                                 1
TUE                                 1
CALC                        Sometimes
MTRANS          Public_Transportation

Output Modelo A: shape (1, 20)
Output Modelo B: shape (1, 18)

IMC esperado: 23.8751
IMC calculado pela pipeline: 23.8751
✓ IMC calculado corretamente dentro da pipeline


---
#### 11. Persistência de artefatos.

1. **`target_encoder.joblib`** — para o app converter int → string ao exibir a predição
2. **Splits processados em `data/processed/`** — o notebook 03 vai carregar sem precisar refazer todo o split

In [14]:
# Persiste o target encoder
target_encoder_path = ARTIFACTS_DIR / TARGET_ENCODER_FILENAME
joblib.dump(target_encoder, target_encoder_path)
print(f'✓ Target encoder salvo em: {target_encoder_path.relative_to(PROJECT_ROOT)}')

# Roundtrip de verificação
te_loaded = joblib.load(target_encoder_path)
assert (te_loaded.transform(splits.y_train) == target_encoder.transform(splits.y_train)).all()
print('✓ Roundtrip do target encoder OK')

✓ Target encoder salvo em: models\artifacts\target_encoder.joblib
✓ Roundtrip do target encoder OK


In [15]:
# Persiste splits para o notebook 03 não precisar refazer tudo
splits_dict = {
    'X_train': splits.X_train, 'y_train': splits.y_train,
    'X_val':   splits.X_val,   'y_val':   splits.y_val,
    'X_test':  splits.X_test,  'y_test':  splits.y_test,
}
splits_path = PROCESSED_DATA_DIR / 'splits.joblib'
joblib.dump(splits_dict, splits_path)
print(f'✓ Splits salvos em: {splits_path.relative_to(PROJECT_ROOT)}')

# Tamanho do arquivo
size_kb = splits_path.stat().st_size / 1024
print(f'  Tamanho: {size_kb:.1f} KB')

✓ Splits salvos em: data\processed\splits.joblib
  Tamanho: 177.4 KB
